# EnergySense - Data Preprocessing and Feature Engineering
This notebook securely loads the original dataset, resamples it, and creates time-series features without data leakage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from features import load_and_preprocess_data, create_features

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:


raw_data_path = r'D:\Users\hp\Projects\ML projects\Energy comsumption project\household_power_consumption.txt'
hourly_df = load_and_preprocess_data(raw_data_path)

print(hourly_df.head())
print(f'Total hours: {len(hourly_df)}')

In [ ]:

plt.figure(figsize=(15, 5))
plt.plot(hourly_df['energy_consumption'].iloc[-1000:], label='Hourly Energy Consumption (kW)')
plt.title('Energy Consumption (Last 1000 hours)')
plt.ylabel('kW')
plt.legend()
plt.show()

In [ ]:


features_df = create_features(hourly_df['energy_consumption'], hourly_df.index, is_training=True)


model_df = pd.concat([hourly_df, features_df], axis=1)


model_df = model_df.dropna()

print(f'Shape after dropping NaNs: {model_df.shape}')
display(model_df.head())

### Verification: Checking for Target Leakage

In [ ]:


idx = model_df.index[10]
prev_idx = model_df.index[9]

print(f'Target at t-1 ({prev_idx}): {model_df.loc[prev_idx, "energy_consumption"]}')
print(f'lag_1 feature at t ({idx}): {model_df.loc[idx, "lag_1"]}')
assert np.isclose(model_df.loc[prev_idx, "energy_consumption"], model_df.loc[idx, "lag_1"])
print('Verification successful: lag_1 correctly represents t-1 data.')

In [ ]:

model_df.to_csv('processed_energy_features.csv')

feature_cols = [col for col in model_df.columns if col != 'energy_consumption']
import joblib
joblib.dump(feature_cols, 'feature_columns.pkl')

print('Saved processed data and feature columns.')

## Stage 3: Baselines & Model Evaluation
Evaluating baselines to ensure the ML model actually learns useful patterns.

In [ ]:

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))


split_idx = int(len(model_df) * 0.8)
train_df = model_df.iloc[:split_idx]
test_df = model_df.iloc[split_idx:]

X_train = train_df.drop('energy_consumption', axis=1)
y_train = train_df['energy_consumption']
X_test = test_df.drop('energy_consumption', axis=1)
y_test = test_df['energy_consumption']

print(f'Train shape: {X_train.shape}')
print(f'Test shape: {X_test.shape}')

In [ ]:


y_pred_persist = X_test['lag_1']

print("Persistence Baseline (One-Step Ahead):")
print(f'MAE: {mean_absolute_error(y_test, y_pred_persist):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_persist)):.4f}')
print(f'WAPE: {wape(y_test, y_pred_persist):.4f}')

In [ ]:


y_pred_seasonal = X_test['lag_24']

print("Seasonal Naive Baseline (One-Step Ahead):")
print(f'MAE: {mean_absolute_error(y_test, y_pred_seasonal):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_seasonal)):.4f}')
print(f'WAPE: {wape(y_test, y_pred_seasonal):.4f}')

In [ ]:

from sklearn.ensemble import HistGradientBoostingRegressor
import time

start_time = time.time()
hgb_model = HistGradientBoostingRegressor(
    max_iter=200,
    learning_rate=0.05,
    max_depth=8,
    random_state=42
)

hgb_model.fit(X_train, y_train)
train_time = time.time() - start_time
print(f'Training completed in {train_time:.2f} seconds')

y_pred_hgb = hgb_model.predict(X_test)

print("HistGradientBoosting Model (One-Step Ahead):")
print(f'MAE: {mean_absolute_error(y_test, y_pred_hgb):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_hgb)):.4f}')
print(f'WAPE: {wape(y_test, y_pred_hgb):.4f}')

In [ ]:

import joblib
joblib.dump(hgb_model, 'final_model.pkl')
print('Saved final_model.pkl (HistGradientBoostingRegressor)')